# MSIT — Fine-tune LLaVA-7B / Qwen-VL / InternLM-XComposer2 (QLoRA)
Reproduction of "Open-World Attribute Mining for E-Commerce Products with
Multimodal Self-Correction Instruction Tuning" (ACL 2025).

**Settings (bắt buộc):**
1. Accelerator: GPU T4 x2 hoặc P100 (Settings → Accelerator)
2. Internet: ON (Settings → Internet)
3. Secrets (Add-ons → Secrets): `HF_TOKEN` (để tải model từ HuggingFace),
   `OPENAI_API_KEY` (tùy chọn — chỉ cần nếu muốn dựng AGTD/CTTD mới bằng GPT-4).
4. (Khuyến nghị) Upload dataset AGTD/CTTD + ảnh sản phẩm dạng Kaggle Dataset
   và gắn vào `/kaggle/input/msit-data/`. Nếu không có, notebook tự dựng
   sample nhỏ từ repo để kiểm chử pipeline end-to-end.

Paper hyperparams được giữ nguyên: Adam, lr=3e-4, 10 epochs (Section 4.1).
Khác biệt duy nhất: QLoRA 4-bit thay vì LoRA 16-bit do giới hạn VRAM 16GB.

In [ ]:
import torch

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
    )
# P100/T4 không hỗ trợ bf16 -> dùng fp16
DTYPE = torch.float16

In [ ]:
!git clone https://github.com/linhnnh688/msit-open-world.git /kaggle/working/msit-open-world
!cd /kaggle/working/msit-open-world && git log --oneline -3

!pip install -q "transformers==4.44.2" "peft==0.13.2" "accelerate==0.34.2" "bitsandbytes==0.43.3" "datasets==3.0.1" safetensors sentencepiece

import sys
sys.path.insert(0, "/kaggle/working/msit-open-world")

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

try:
    secrets = UserSecretsClient()
    HF_TOKEN = secrets.get_secret("HF_TOKEN")
    OPENAI_API_KEY = secrets.get_secret("OPENAI_API_KEY")  # optional
except Exception as e:
    print("Secrets warning:", e)
    HF_TOKEN = os.environ.get("HF_TOKEN", "")
    OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")

if HF_TOKEN:
    from huggingface_hub import login

    login(token=HF_TOKEN)

from msit.config import MSITConfig

cfg = MSITConfig()

# Cho phép thu nhỏ quy mô khi chạy thử trên Kaggle (paper: 1000 AGTD + 300 CTTD)
MAX_AGTD = int(os.environ.get("MAX_AGTD", "1000"))
MAX_CTTD = int(os.environ.get("MAX_CTTD", "300"))
EPOCHS = int(os.environ.get("EPOCHS", str(cfg.epochs)))  # paper: 10
print(
    f"lr={cfg.learning_rate}, epochs={EPOCHS}, MAX_AGTD={MAX_AGTD}, MAX_CTTD={MAX_CTTD}"
)

In [ ]:
import json, glob, os
from msit.models.mock_backend import MockMLLMBackend

DATA_DIR = "/kaggle/input/msit-data"  # Kaggle Dataset do bạn upload (tùy chọn)
agtd_path = os.path.join(DATA_DIR, "agtd.jsonl")
cttd_path = os.path.join(DATA_DIR, "cttd.jsonl")


def load_jsonl(p):
    if os.path.exists(p):
        with open(p, encoding="utf-8") as f:
            return [json.loads(l) for l in f if l.strip()]
    return []


agtd = load_jsonl(agtd_path)[:MAX_AGTD]
cttd = load_jsonl(cttd_path)[:MAX_CTTD]

if not agtd and not cttd:
    print(
        "[warn] Không tìm thấy AGTD/CTTD trong /kaggle/input/msit-data — "
        "dựng sample nhỏ từ repo để kiểm chứng pipeline."
    )
    with open(
        "/kaggle/working/msit-open-world/msit/data_samples/sample_products.json"
    ) as f:
        prods = json.load(f)["products"]
    db = {p["title"]: p["gold"] for p in prods}
    from msit.data.agtd_builder import AGTDBuilder
    from msit.data.cttd_builder import CTTDBuilder

    agtd = AGTDBuilder(MockMLLMBackend(db)).build_batch(prods)
    cttd = CTTDBuilder(MockMLLMBackend(db)).build_batch(
        [{**p, "attributes": p["gold"]} for p in prods], attributes_per_product=2
    )

train_data = agtd + cttd
print(f"AGTD={len(agtd)}, CTTD={len(cttd)}, total={len(train_data)}")
print(json.dumps(train_data[0], ensure_ascii=False, default=str)[:500])

In [ ]:
import torch, random
from dataclasses import dataclass
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
from transformers import (
    AutoModelForVision2Seq,
    AutoModelForCausalLM,
    AutoTokenizer,
    AutoProcessor,
    Trainer,
    TrainingArguments,
    BitsAndBytesConfig,
)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=DTYPE,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)


def load_model_and_processor(model_id: str, vision2seq: bool):
    loader = AutoModelForVision2Seq if vision2seq else AutoModelForCausalLM
    model = loader.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        torch_dtype=DTYPE,
        device_map="auto",
        trust_remote_code=not vision2seq,
        token=HF_TOKEN or None,
    )
    model = prepare_model_for_kbit_training(model)
    model.config.use_cache = False
    model.gradient_checkpointing_enable()
    tok_cls = AutoProcessor if vision2seq else AutoTokenizer
    processor = tok_cls.from_pretrained(
        model_id, trust_remote_code=not vision2seq, token=HF_TOKEN or None
    )
    return model, processor


def make_collator(processor, vision2seq: bool):
    tok = processor.tokenizer if vision2seq else processor

    def collate(batch):
        input_ids_list, labels_list, pixel_values = [], [], []
        for item in batch:
            prompt = f"{item['instruction']}\n\n{item['input']}"
            p_ids = tok(prompt, add_special_tokens=False)["input_ids"]
            c_ids = tok(item["output"] + tok.eos_token, add_special_tokens=False)[
                "input_ids"
            ]
            input_ids_list.append(p_ids + c_ids)
            labels_list.append([-100] * len(p_ids) + c_ids)
        maxlen = max(len(x) for x in input_ids_list)
        pad_id = tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id
        input_ids, labels, attn = [], [], []
        for ids, lab in zip(input_ids_list, labels_list):
            pad = maxlen - len(ids)
            input_ids.append(ids + [pad_id] * pad)
            labels.append(lab + [-100] * pad)
            attn.append([1] * len(ids) + [0] * pad)
        out = {
            "input_ids": torch.tensor(input_ids),
            "attention_mask": torch.tensor(attn),
            "labels": torch.tensor(labels),
        }
        return out

    return collate


def train_msit_lora(
    model_id: str, vision2seq: bool, output_dir: str, train_data, lora_target_modules
):
    model, processor = load_model_and_processor(model_id, vision2seq)
    model = get_peft_model(
        model,
        LoraConfig(
            task_type=TaskType.CAUSAL_LM,
            r=cfg.lora_r,
            lora_alpha=cfg.lora_alpha,
            lora_dropout=cfg.lora_dropout,
            target_modules=lora_target_modules,
            bias="none",
        ),
    )
    model.print_trainable_parameters()

    args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=EPOCHS,  # paper: 10
        learning_rate=cfg.learning_rate,  # paper: 3e-4
        optim="adamw_torch",  # paper: Adam
        per_device_train_batch_size=2,  # nhỏ để vừa 16GB VRAM
        gradient_accumulation_steps=8,  # effective batch ~16
        logging_steps=5,
        save_strategy="no",
        fp16=True,
        report_to=[],
        seed=cfg.random_seed,
        gradient_checkpointing=True,
    )
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_data,
        data_collator=make_collator(processor, vision2seq),
    )
    trainer.train()
    model.save_pretrained(output_dir)
    processor.save_pretrained(output_dir)
    del model, trainer
    torch.cuda.empty_cache()
    print("saved ->", output_dir)

In [ ]:
# MSIT(LLaVA-7B) — dùng checkpoint llava-hf tương thích transformers
train_msit_lora(
    model_id="llava-hf/llava-1.5-7b-hf",
    vision2seq=True,
    output_dir="/kaggle/working/adapters/msit-llava-7b",
    train_data=train_data,
    lora_target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
)

In [ ]:
# MSIT(Qwen) — Qwen-VL cần trust_remote_code
train_msit_lora(
    model_id="Qwen/Qwen-VL-Chat",
    vision2seq=False,
    output_dir="/kaggle/working/adapters/msit-qwen-vl-7b",
    train_data=train_data,
    lora_target_modules=["c_attn", "c_proj"],
)

In [ ]:
# MSIT(InternLM)
train_msit_lora(
    model_id="internlm/internlm-xcomposer2-7b",
    vision2seq=False,
    output_dir="/kaggle/working/adapters/msit-internlm-xc2-7b",
    train_data=train_data,
    lora_target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
)

In [ ]:
!cd /kaggle/working && zip -qr msit_adapters.zip adapters && ls -lh msit_adapters.zip

# Smoke test: load adapter LLaVA và chạy 1 sample Stage-1-style
from peft import PeftModel
model, processor = load_model_and_processor("llava-hf/llava-1.5-7b-hf", vision2seq=True)
model = PeftModel.from_pretrained(model, "/kaggle/working/adapters/msit-llava-7b")
sample = train_data[0]
prompt = f"{sample['instruction']}\n\n{sample['input']}"
inputs = processor(text=prompt, return_tensors="pt").to(model.device)
out = model.generate(**inputs, max_new_tokens=200, do_sample=True,
                     temperature=cfg.temperature, top_p=cfg.top_p)
print(processor.decode(out[0], skip_special_tokens=True))